# 📚 18. Revisión de literatura y evidencia de novedad

La guía exige **demostrar** la novedad con una búsqueda documentada, no declararla. Este capítulo reporta la búsqueda (bases, cadenas, fecha y resultados), la matriz de novedad, los trabajos más cercanos (por aplicación y por familia algorítmica) y una declaración de novedad acotada a lo que la búsqueda permite afirmar.

## 18.1 Protocolo

- **Bases consultadas (4):** OpenAlex (índice abierto con más de 250 millones de trabajos, sucesor de Microsoft Academic), Semantic Scholar, arXiv y Crossref. Todas se consultaron por su API pública con el script `book/revision_literatura.py`, que guarda el conteo y los 10 primeros resultados de cada búsqueda (`resultados/literatura_conteos.csv` y `literatura_resultados.csv`).
- **Limitaciones de acceso (declaradas):** no se consultaron **Scopus ni Web of Science** por no tener acceso institucional desde el entorno de trabajo; la guía los menciona como ejemplos, no como obligatorios. **DBLP** se intentó y bloqueó el acceso automatizado. **Semantic Scholar** limitó las peticiones (error 429) en la mayoría de las cadenas, aun con reintentos espaciados; las que fallaron quedan registradas como fallidas. El **conteo de Crossref** no es comparable con el de las otras bases, porque cuenta cualquier coincidencia parcial de términos; de Crossref se usan solo sus 10 primeros resultados (ordenados por relevancia).
- **Cadenas (10):** combinan (i) la familia del modelo (combinación de pronósticos, pesos en el símplex, árboles / particionamiento recursivo, selección dinámica, meta-aprendizaje), (ii) el problema (ventas diarias de estaciones de servicio y tiendas de conveniencia) y (iii) sinónimos en inglés y español.
- **Revisión:** título de los 10 primeros resultados de cada base y cadena; **resumen** de los 14 trabajos más cercanos (tabla 18.3).

In [1]:
import json
import pandas as pd
pd.set_option("display.max_colwidth", 90)
conteos = pd.read_csv("resultados/literatura_conteos.csv")
res = pd.read_csv("resultados/literatura_resultados.csv")
print("fecha de la consulta:", conteos["fecha"].iloc[0])
conteos.drop_duplicates("id")[["id", "cadena"]].set_index("id")

fecha de la consulta: 2026-10-24


,cadena
id,
C1,"""forecast combination"" regime tree ""sales"""
C2,"""forecast combination"" simplex weights decision tree"
C3,"""model-based recursive partitioning"" forecast combination"
C4,"""dynamic ensemble selection"" ""sales forecasting"""
C5,"""gas station"" OR ""fuel station"" daily sales forecasting machine learning"
C6,"""convenience store"" daily sales forecasting ensemble"
C7,"""combinación de pronósticos"" ventas diarias"
C8,"""pronóstico de ventas"" ""estaciones de servicio"""
C9,meta-learning forecast combination weights features FFORMA


## 18.2 Matriz de novedad

**Relevantes** cuenta los resultados, entre los revisados, que usan una combinación de pronósticos con pesos dependientes del estado/régimen **o** que pronostican ventas diarias de estaciones de servicio o tiendas de conveniencia. La observación resume el resultado más cercano.

In [2]:
relevantes = {  # (cadena, base): (n relevantes, observación)
    ("C1", "OpenAlex"): (1, "Tesis (2010) sobre cuándo conviene combinar pronósticos; sin pesos por régimen ni ventas de estaciones"),
    ("C1", "Crossref"): (1, "Elliott y Timmermann (2005): pesos que cambian con un régimen latente (Markov), macroeconomía"),
    ("C2", "OpenAlex"): (2, "Combinación dinámica por aprendizaje por refuerzo (Fu et al., 2022); combinación estructurada en energía"),
    ("C2", "Crossref"): (3, "Tetereva y Kazak (2026): árboles con pesos constantes por régimen para VaR/ES; encogimiento de pesos"),
    ("C3", "OpenAlex"): (1, "SETAR-Tree (Godahewa et al., 2023): árbol por umbrales con modelos AR en las hojas, no combinaciones"),
    ("C3", "Crossref"): (0, "Particionamiento recursivo basado en modelos aplicado a otros dominios"),
    ("C4", "OpenAlex"): (1, "Selección dinámica de ensambles para ventas por telemarketing (clasificación)"),
    ("C4", "Crossref"): (1, "Selección dinámica de modelos por patrón de demanda en retail (E et al., 2022)"),
    ("C5", "OpenAlex"): (2, "Ventas de gasolineras con BiLSTM + algoritmo genético (Pan et al., 2022); ARIMA–Markov en 150 estaciones (2025)"),
    ("C5", "Crossref"): (1, "SARIMA mensual de ventas de gasolina en una estación de Zimbabue (2026)"),
    ("C6", "OpenAlex"): (1, "Selección dinámica por patrón de demanda en retail"),
    ("C6", "Crossref"): (1, "Sistema híbrido de pronóstico en tiendas de conveniencia (2011, retractado)"),
    ("C7", "OpenAlex"): (1, "Combinación bayesiana con metaheurística (caso de estudio, 2018), sin régimen"),
    ("C8", "OpenAlex"): (0, "Planes de negocio; ningún pronóstico de ventas de estaciones de servicio en español"),
    ("C9", "OpenAlex"): (2, "Combinación bayesiana con variables que cambian en el tiempo (Li et al., 2022); meta-aprendizaje de ensambles"),
    ("C9", "Crossref"): (2, "Meta-aprendizaje para combinación (Lemke y Gabrys, 2010); pesos por horizonte"),
    ("C10", "OpenAlex"): (1, "Pronóstico por grupos de clientes con promedio bayesiano de modelos (Seyedan et al., 2022)"),
    ("C10", "Crossref"): (1, "Pesos negativos y encogimiento en combinación de pronósticos"),
}
mat = conteos.copy()
mat["relevantes"] = [relevantes.get((i, b), (0, ""))[0] for i, b in zip(mat["id"], mat["base"])]
mat["observación"] = [relevantes.get((i, b), (0, "Sin resultados relacionados"))[1] for i, b in zip(mat["id"], mat["base"])]
mat.loc[mat["resultados"].isna(), "observación"] = "Consulta fallida (límite de peticiones del API, error 429)"
mat.loc[mat["resultados"].isna(), "relevantes"] = None
mat.loc[(mat["base"] == "Crossref"), "resultados"] = mat.loc[(mat["base"] == "Crossref"), "resultados"].map(lambda v: f"{v:,.0f}*")
mat = mat.rename(columns={"base": "base de datos", "cadena": "cadena de búsqueda", "revisados": "revis."})
mat[["id", "base de datos", "cadena de búsqueda", "resultados", "revis.", "relevantes", "observación"]].set_index(["id", "base de datos"])

/var/folders/51/pqfk6vq57gsbrjlv4ycg71j00000gn/T/ipykernel_9164/1647371610.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['796,903*' '1,427,047*' '9,714,157*' '1,772,279*' '4,823,811*' '644,698*'
 '10,338,666*' '10,339,983*' '3,987,688*' '2,508,160*']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mat.loc[(mat["base"] == "Crossref"), "resultados"] = mat.loc[(mat["base"] == "Crossref"), "resultados"].map(lambda v: f"{v:,.0f}*")


cadena de búsqueda  \
id  base de datos                                                                                
C1  OpenAlex                                        "forecast combination" regime tree "sales"   
    Semantic Scholar                                "forecast combination" regime tree "sales"   
    arXiv                                           "forecast combination" regime tree "sales"   
    Crossref                                        "forecast combination" regime tree "sales"   
C2  OpenAlex                              "forecast combination" simplex weights decision tree   
    Semantic Scholar                      "forecast combination" simplex weights decision tree   
    arXiv                                 "forecast combination" simplex weights decision tree   
    Crossref                              "forecast combination" simplex weights decision tree   
C3  OpenAlex                         "model-based recursive partitioning" forecast combination   
    Semantic Scholar                 "model-based recursive partitioning" forecast combination   
    arXiv                            "model-based recursive partitioning" forecast combination   
    Crossref                         "model-based recursive partitioning" forecast combination   
C4  OpenAlex                                  "dynamic ensemble selection" "sales forecasting"   
    Semantic Scholar                          "dynamic ensemble selection" "sales forecasting"   
    arXiv                                     "dynamic ensemble selection" "sales forecasting"   
    Crossref                                  "dynamic ensemble selection" "sales forecasting"   
C5  OpenAlex          "gas station" OR "fuel station" daily sales forecasting machine learning   
    Semantic Scholar  "gas station" OR "fuel station" daily sales forecasting machine learning   
    arXiv             "gas station" OR "fuel station" daily sales forecasting machine learning   
    Crossref          "gas station" OR "fuel station" daily sales forecasting machine learning   
C6  OpenAlex                              "convenience store" daily sales forecasting ensemble   
    Semantic Scholar                      "convenience store" daily sales forecasting ensemble   
    arXiv                                 "convenience store" daily sales forecasting ensemble   
    Crossref                              "convenience store" daily sales forecasting ensemble   
C7  OpenAlex                                       "combinación de pronósticos" ventas diarias   
    Semantic Scholar                               "combinación de pronósticos" ventas diarias   
    arXiv                                          "combinación de pronósticos" ventas diarias   
    Crossref                                       "combinación de pronósticos" ventas diarias   
C8  OpenAlex                                   "pronóstico de ventas" "estaciones de servicio"   
    Semantic Scholar                           "pronóstico de ventas" "estaciones de servicio"   
    arXiv                                      "pronóstico de ventas" "estaciones de servicio"   
    Crossref                                   "pronóstico de ventas" "estaciones de servicio"   
C9  OpenAlex                        meta-learning forecast combination weights features FFORMA   
    Semantic Scholar                meta-learning forecast combination weights features FFORMA   
    arXiv                           meta-learning forecast combination weights features FFORMA   
    Crossref                        meta-learning forecast combination weights features FFORMA   
C10 OpenAlex             stacking non-negative weights partition "forecast combination" retail   
    Semantic Scholar     stacking non-negative weights partition "forecast combination" retail   
    arXiv                stacking non-negative weights partition "forecast combination" retail   
    Crossref             stacking non-negative weights par

\* Crossref cuenta coincidencias parciales de cualquier término; se revisan sus 10 primeros por relevancia.

## 18.3 Trabajos más cercanos y diferencias

Se leyó el resumen de los trabajos más cercanos, por **aplicación** (ventas de estaciones de servicio y retail) y por **familia algorítmica** (combinación de pronósticos con pesos que dependen del estado, árboles y particionamiento, meta-aprendizaje).

In [3]:
cercanos = pd.DataFrame([
    ["Tetereva y Kazak (2026), SSRN", "árboles de combinación por régimen",
     "Pesos de combinación constantes dentro de regímenes definidos por un procedimiento de árboles sobre variables macro-financieras; pérdida de VaR/ES; varios árboles con muestreo de variables por grupos",
     "Misma idea de regímenes por árbol. El ACS: pérdida cuadrática y pesos en el símplex resueltos con gradiente proyectado sobre la matriz de errores (búsqueda de cortes sin depender de n); encogimiento jerárquico hacia el padre y hacia el promedio; desacuerdo entre expertos como variable de régimen; ajuste cruzado; aplicación a ventas diarias"],
    ["Elliott y Timmermann (2005), IER", "combinación con cambio de régimen",
     "Pesos que dependen de un estado latente de Markov estimado con el modelo de cambio de régimen", "Régimen observable (variables del día anterior) y partición por árbol, no un estado latente; sin modelo generativo"],
    ["Montero-Manso et al. (2020), IJF — FFORMA", "meta-aprendizaje",
     "XGBoost aprende pesos (softmax) para cada serie a partir de rasgos de la serie completa", "Pesos por negocio-día (no por serie), exactos y convexos por hoja, interpretables"],
    ["Li, Kang y Li (2022), IJF", "combinación bayesiana con rasgos variables",
     "Pesos que cambian en el tiempo según rasgos de la serie, con inferencia bayesiana", "Partición discreta y legible; solver convexo propio; sin MCMC"],
    ["van der Laan et al. (2007) — Super Learner; Breiman (1996)", "stacking con pesos no negativos",
     "Pesos globales no negativos (o convexos) estimados con validación cruzada", "Pesos distintos por región del espacio de régimen; se compara en la ablación (pesos globales en el símplex)"],
    ["Zeileis et al. (2008) — MOB; Quinlan (1992) — M5", "árboles de modelos",
     "Árboles cuyas hojas tienen un modelo paramétrico (lineal)", "Las hojas son combinaciones convexas de expertos, con encogimiento al padre"],
    ["Godahewa et al. (2023), ML — SETAR-Tree", "árboles por umbrales para series",
     "Árbol global con modelos autorregresivos en las hojas", "Hojas que combinan modelos ya entrenados; no reemplaza a los expertos"],
    ["E, Yu y Tian (2022), Mathematics", "selección dinámica de modelos en retail",
     "Clasifica cada producto por patrón de demanda y elige o combina modelos según el desempeño por clase", "Régimen por día (no por producto), aprendido, y pesos óptimos por hoja"],
    ["Seyedan et al. (2022), DAJ", "promedio bayesiano por grupos",
     "Pronóstico por segmento de clientes y combinación por BMA", "Pesos por régimen del día del negocio, no por segmento fijo"],
    ["Pan, Liao y Liang (2022), Petroleum Science", "aplicación: ventas de gasolineras",
     "BiLSTM multivariable con hiperparámetros por algoritmo genético; supera a XGBoost y ARIMA", "Modelo no neuronal, por negocio-día, para EDS + tiendas + kioscos con datos por medio de pago"],
    ["Kubek y Więcek (2025), Energies", "aplicación: demanda en estaciones de combustible",
     "ARIMA con cadenas de Markov en ~150 series de estaciones", "Combinación de modelos de ML con pesos por régimen; venta total en pesos, no volumen"],
], columns=["trabajo", "familia", "qué hace", "qué es distinto en el ACS"])
cercanos.set_index("trabajo")

,familia,qué hace,qué es distinto en el ACS
trabajo,,,
"Tetereva y Kazak (2026), SSRN",árboles de combinación por régimen,Pesos de combinación constantes dentro de regímenes definidos por un procedimiento de ...,Misma idea de regímenes por árbol. El ACS: pérdida cuadrática y pesos en el símplex re...
"Elliott y Timmermann (2005), IER",combinación con cambio de régimen,Pesos que dependen de un estado latente de Markov estimado con el modelo de cambio de ...,"Régimen observable (variables del día anterior) y partición por árbol, no un estado la..."
"Montero-Manso et al. (2020), IJF — FFORMA",meta-aprendizaje,XGBoost aprende pesos (softmax) para cada serie a partir de rasgos de la serie completa,"Pesos por negocio-día (no por serie), exactos y convexos por hoja, interpretables"
"Li, Kang y Li (2022), IJF",combinación bayesiana con rasgos variables,"Pesos que cambian en el tiempo según rasgos de la serie, con inferencia bayesiana",Partición discreta y legible; solver convexo propio; sin MCMC
van der Laan et al. (2007) — Super Learner; Breiman (1996),stacking con pesos no negativos,Pesos globales no negativos (o convexos) estimados con validación cruzada,Pesos distintos por región del espacio de régimen; se compara en la ablación (pesos gl...
Zeileis et al. (2008) — MOB; Quinlan (1992) — M5,árboles de modelos,Árboles cuyas hojas tienen un modelo paramétrico (lineal),"Las hojas son combinaciones convexas de expertos, con encogimiento al padre"
"Godahewa et al. (2023), ML — SETAR-Tree",árboles por umbrales para series,Árbol global con modelos autorregresivos en las hojas,Hojas que combinan modelos ya entrenados; no reemplaza a los expertos
"E, Yu y Tian (2022), Mathematics",selección dinámica de modelos en retail,Clasifica cada producto por patrón de demanda y elige o combina modelos según el desem...,"Régimen por día (no por producto), aprendido, y pesos óptimos por hoja"
"Seyedan et al. (2022), DAJ",promedio bayesiano por grupos,Pronóstico por segmento de clientes y combinación por BMA,"Pesos por régimen del día del negocio, no por segmento fijo"


## 18.4 Declaración de novedad

```{important}
Al mejor de nuestro conocimiento y según la búsqueda descrita (4 bases, 10 cadenas, consulta del 24 de octubre de 2026), **no se encontraron estudios que apliquen un árbol de combinación de pronósticos con pesos en el símplex por régimen al pronóstico de la venta diaria de estaciones de servicio, tiendas de conveniencia y kioscos.** Los trabajos de aplicación más cercanos pronostican ventas de gasolineras con redes neuronales (Pan et al., 2022), ARIMA–Markov (Kubek y Więcek, 2025) o SARIMA mensual (2026), sin combinación por régimen.

**Lo que no es nuevo.** La idea de que los pesos de combinación dependan de un régimen existe desde Elliott y Timmermann (2005), y **Tetereva y Kazak (2026)** ya usan árboles para definir regímenes con pesos constantes dentro de cada uno (para riesgo de cola financiero). El ACS **no** reclama la idea general.

**Lo que sí aporta el ACS** (mecanismo, no solo aplicación): (1) pesos en el símplex con pérdida cuadrática, resueltos con un gradiente proyectado propio sobre la **matriz de errores** de los expertos, cuya búsqueda de cortes con sumas acumuladas **no depende de n**; (2) **encogimiento jerárquico** de cada hoja hacia su padre y de la raíz hacia el promedio simple, con garantía de entrenamiento (Proposición 4); (3) el **desacuerdo entre expertos** como variable de régimen; (4) **ajuste cruzado** de los pronósticos fuera de muestra para no subestimar a los expertos que más mejoran con datos.
```

**Condición R1 (novedad en la aplicación).** Se cumple según la búsqueda: no se encontró el modelo aplicado a este fenómeno y tarea. La búsqueda tiene límites declarados (sin Scopus ni Web of Science; Semantic Scholar parcial), así que la afirmación es "al mejor de nuestro conocimiento" y se acota a esas bases.